In [45]:
import ast
import glob
from collections import defaultdict
from datetime import date

import wandb
import numpy as np
import pandas as pd

# Set display options to show all columns and format floats nicely
pd.set_option('display.max_columns', None)
#pd.set_option('display.float_format', '{:.2f}'.format)

user = "imos-buildingenergy-1"


today = date.today()
api = wandb.Api()

# # Find all csv files in the current directory
csv_files = glob.glob("*.csv")
# # Collect all the names of the csv files without the extension
csv_names = [csv_file[:-4] for csv_file in csv_files]
project_name = "TabPHM-autoencoders-checks_02_10_night"
 

if project_name not in csv_names:
    runs = api.runs(f"{user}/{project_name}")

    summary_list, config_list, name_list = [], [], []
    for run in runs:
        # ✅ Add this check to skip runs that are not finished
        if run.state == "finished":
            # .summary contains the output keys/values for metrics like accuracy.
            #  We call ._json_dict to omit large files
            summary_list.append(run.summary._json_dict)

            # .config contains the hyperparameters.
            #  We remove special values that start with _.
            config_list.append(
                {k: v for k, v in run.config.items() if not k.startswith("_")}
            )

            # .name is the human-readable name of the run.
            name_list.append(run.name)
        else:
            print(f"Skipping run {run.name} with state {run.state}")

    runs_df = pd.DataFrame(
        {"summary": summary_list, "config": config_list, "run_name": name_list}
    )

    runs_df.to_csv(f"csv_files/{project_name}.csv")
else:
    runs_df = pd.read_csv(f"csv_files/{project_name}.csv", index_col=0)

    for row in runs_df.iloc:
        row["summary"] = ast.literal_eval(row["summary"])
        row["config"] = ast.literal_eval(row["config"])


for row in runs_df.iloc:
    row["summary"].update(row["config"])

lst = [
    {**i["summary"], "run_name": i["run_name"]}
    for i in runs_df.iloc
]
df = pd.DataFrame.from_dict(lst)

df_init = df.copy()

# Get average epoch run time
df["epoch_run_time"] = df["_runtime"] / df["epoch"]



Skipping run pronostia+reconstruction+residual_autoencoder_8e1f2977 with state running
Skipping run pronostia+reconstruction+residual_autoencoder_f129008e with state running


### Go over different config groups and flatten the content from the nested dictionaries into separate columns

In [46]:
def normalize_column(df, column_to_normalize):
    # Use json_normalize to flatten the nested dictionaries into separate columns
    flattened_df = pd.json_normalize(df[column_to_normalize])
    # Rename columns to include 'nested_column' prefix
    flattened_df.columns = [
        f"{column_to_normalize}.{col}" for col in flattened_df.columns
    ]
    # Concatenate the flattened DataFrame with the original DataFrame
    result_df = pd.concat([df, flattened_df], axis=1)
    # Get new columns names
    new_columns = flattened_df.columns
    # Drop the original nested column if needed
    result_df.drop(column_to_normalize, axis=1, inplace=True)
    return result_df, new_columns


# Config columns to normalize
columns_to_normalize = ["optimizer", "datasource", "task_definition","model", "dataset", "datamodule", "callbacks", "paths"]
droped_columns = []
# Keep track of config columns added
config_columns = []
for column in columns_to_normalize:
    df, columns = normalize_column(df, column)
    config_columns.extend(columns)

In [47]:


# Describe only the numerical columns
print("--- 📊 Numerical Columns Summary ---")
display(df.describe(include='number'))

# Describe only the object/categorical columns
print("\n--- 📝 Object/Categorical Columns Summary ---")
display(df.describe(include=['object']))

# check columns for nans and provide statistics
nan_stats = df.isna().sum()
nan_stats = nan_stats[nan_stats > 0]
print(f"\n--- ⚠️ Columns with NaN Values, {(nan_stats.shape[0]/ len(df.columns)) *100}% has NaN values ---")

# for every column calucalate also the percentage of nans
nan_stats = pd.DataFrame(nan_stats, columns=["num_nans"])
nan_stats["percentage_nans"] = (nan_stats["num_nans"] / len(df)) * 100
nan_stats = nan_stats.sort_values(by="percentage_nans", ascending=False)
display(nan_stats)


# Take all the columns that all more the 90% nans
nan_cols_to_drop = nan_stats[nan_stats["percentage_nans"] > 90].index.tolist()
print(f"Dropping {len(nan_cols_to_drop)} columns with more than 90% NaNs")
# Remove columns that are not needed (we shouldn't vary them or their variation is not interesting)
df = df.drop(nan_cols_to_drop, axis=1)
droped_columns.extend(nan_cols_to_drop)

--- 📊 Numerical Columns Summary ---


/home/levtel/miniconda3/envs/phm2/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,_runtime,_step,_timestamp,epoch,lr-Adam,train/loss_step,trainer/global_step,seed,num_workers,val_batch_size,prefetch_factor,test_batch_size,train_batch_size,model/params/total,model/params/trainable,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,test/loss,test/mae,test/mse,test/rmse,train/loss_epoch,train/mae,train/mse,train/rmse,val/loss,val/mae,val/mse,val/rmse,epoch_run_time,optimizer.scheduler.scheduler_params.gamma,optimizer.scheduler.scheduler_params.step_size,optimizer.parameters.lr,optimizer.parameters.weight_decay,datasource.fold,task_definition.stride,task_definition.seq_len,task_definition.pred_len,task_definition.label_len,model.latent_dim,model.dropout_prob,model.input_seq_len,model.input_channels,dataset.dataset_cfg.stride,dataset.dataset_cfg.seq_len,dataset.dataset_cfg.pred_len,dataset.dataset_cfg.label_len,datamodule.num_workers,datamodule.val_batch_size,datamodule.prefetch_factor,datamodule.test_batch_size,datamodule.train_batch_size,callbacks.model_summary.max_depth,callbacks.early_stopping.patience,callbacks.early_stopping.min_delta,callbacks.model_checkpoint.save_top_k
count,56.000000,56.000000,5.600000e+01,56.000000,56.000000,5.600000e+01,5.600000e+01,56.0,56.0,56.000000,56.0,56.000000,56.000000,5.600000e+01,5.600000e+01,52.000000,52.000000,52.000000,52.000000,52.000000,52.000000,52.000000,52.000000,5.200000e+01,52.000000,5.200000e+01,52.000000,5.200000e+01,52.000000,5.200000e+01,52.000000,5.200000e+01,52.000000,5.200000e+01,52.000000,56.0000,56.0,56.000000,56.000000,56.0,4.0,56.0,56.0,56.0,56.0,56.000000,5.600000e+01,56.0,56.000000,56.0,56.0,56.0,56.0,56.0,56.000000,56.0,56.000000,56.000000,56.0,56.0,56.0,56.0
mean,3716.964286,3768.785714,1.759389e+09,18.571429,0.003714,2.200179e+03,1.837292e+05,42.0,4.0,988.000000,10.0,988.000000,422.142857,7.386723e+05,7.386723e+05,0.017455,0.003139,195.498015,5.185852,0.007865,0.001259,8.371936,0.195446,1.365862e+03,4.615768,4.096724e+03,6.206439,1.233731e+06,15.525602,3.696754e+06,235.662389,1.365862e+03,4.615768,4.096724e+03,6.206439,inf,0.5,14.642857,0.004714,0.0,0.0,1.0,64.0,0.0,0.0,121.142857,2.000000e-01,64.0,2.928571,1.0,64.0,0.0,0.0,4.0,988.000000,10.0,988.000000,422.142857,-1.0,25.0,0.0,1.0
std,7677.152498,7871.845723,2.144351e+04,5.197402,0.004023,1.603093e+04,3.934225e+05,0.0,0.0,188.754386,0.0,188.754386,351.615463,4.044015e+05,4.044015e+05,0.001542,0.000945,391.869440,6.231972,0.001002,0.000999,19.047491,0.431090,9.737354e+03,28.329642,2.921217e+04,36.788148,6.267359e+06,68.262730,1.880278e+07,1096.035091,9.737354e+03,28.329642,2.921217e+04,36.788148,NaN,0.0,5.032363,0.004623,0.0,0.0,0.0,0.0,0.0,0.0,24.947529,2.800676e-17,0.0,0.259870,0.0,0.0,0.0,0.0,0.0,188.754386,0.0,188.754386,351.615463,0.0,0.0,0.0,0.0
min,20.000000,21.000000,1.759358e+09,0.000000,0.000250,6.336087e-08,1.049000e+03,42.0,4.0,16.000000,10.0,16.000000,16.000000,1.832080e+05,1.832080e+05,0.015781,0.001872,28.311021,0.737676,0.006267,0.000581,2.455972,0.030099,9.187757e-08,0.000179,2.756327e-07,0.000303,2.105937e-07,0.000272,6.317774e-07,0.000459,9.187757e-08,0.000179,2.756327e-07,0.000303,30.2000,0.5,10.000000,0.000500,0.0,0.0,1.0,64.0,0.0,0.0,32.000000,2.000000e-01,64.0,2.000000,1.0,64.0,0.0,0.0,4.0,16.000000,10.0,16.000000,16.000000,-1.0,25.0,0.0,1.0
25%,706.500000,544.000000,1.759378e+09,20.000000,0.000500,1.942652e-05,2.212000e+04,42.0,4.0,1024.000000,10.0,1024.000000,128.000000,3.801720e+05,3.801720e+05,0.016365,0.002378,46.428217,2.536142,0.007000,0.000857,2.699031,0.052178,1.280439e-05,0.002417,3.841314e-05,0.003555,1.682006e-05,0.002833,5.046017e-05,0.003973,1.280439e-05,0.002417,3.841314e-05,0.003555,52.3375,0.5,10.000000,0.000500,0.0,0.0,1.0,64.0,0.0,0.0,128.000000,2.000000e-01,64.0,3.000000,1.0,64.0,0.0,0.0,4.0,1024.000000,10.0,1024.000000,128.000000,-1.0,25.0,0.0,1.0
50%,1462.500000,1428.000000,1.759391e+09,20.000000,0.001000,2.257032e-03


--- 📝 Object/Categorical Columns Summary ---


,_wandb,loss,tags,cache,logger,trainer,exp_name,ckpt_path,evaluator,hydra_cfg,transforms,profiler_cfg,subset_range,experiment_group,run_name,plots/test/reconstructions_overview,plots/train/reconstructions_overview,plots/val/reconstructions_overview,optimizer._target_,optimizer.optimizer_id,optimizer.scheduler.scheduler_id,datasource._target_,datasource.data_dir,datasource.data_name,datasource.task_mode,datasource.cache_dir,task_definition.task_type,task_definition.model.data_requirements.input_tensors,model.kernels,model.strides,model._target_,model.dilations,model.output_channels,model.blocks_config,dataset._target_,dataset.dataset_cfg._target_,datamodule._target_,callbacks.model_timer._target_,callbacks.model_summary._target_,callbacks.early_stopping.mode,callbacks.early_stopping.monitor,callbacks.early_stopping._target_,callbacks.early_stopping.stopping_threshold,callbacks.early_stopping.divergence_threshold,callbacks.early_stopping.check_on_train_epoch_end,callbacks.model_checkpoint.mode,callbacks.model_checkpoint.dirpath,callbacks.model_checkpoint.monitor,callbacks.model_checkpoint._target_,callbacks.model_checkpoint.filename,callbacks.model_checkpoint.every_n_epochs,callbacks.model_checkpoint.every_n_train_steps,callbacks.model_checkpoint.train_time_interval,callbacks.model_checkpoint.save_on_train_epoch_end,callbacks.learning_rate_monitor._target_,callbacks.learning_rate_monitor.logging_interval,paths.log_dir,paths.ckpt_dir,paths.data_dir,paths.plot_dir,paths.root_dir,paths.work_dir,paths.cache_path,paths.output_dir,paths.eval_details,paths.artifacts_dir,paths.model_workdir,paths.model_cache_dir
count,56,56,56,56,56,56,56,0,56,56,56,0,0,56,56,52,52,52,56,56,56,56,52,56,56,4,56,56,56,56,56,56,56,4,56,56,56,56,56,56,56,56,0,0,0,56,56,56,56,56,0,0,0,0,56,56,56,56,56,56,56,56,56,56,56,56,56,56
unique,53,1,1,1,56,53,2,0,53,56,1,0,0,1,56,52,52,52,1,1,1,2,1,2,2,1,1,1,2,2,1,2,1,1,1,1,1,1,1,1,1,1,0,0,0,1,53,1,1,1,0,0,0,0,1,1,53,53,1,53,1,1,2,53,53,1,53,53
top,{'runtime': 1875},{'_target_': 'picid.loss.default.MSELoss'},[dev],"{'use_cache_after_loading': True, 'use_cache_a...",{'wandb': {'id': 'unibo+reconstruction+residua...,"{'devices': [1], '_target_': 'lightning.pytorc...",unibo/reconstruction/residual_autoencoder,NaN,{'val': {'paths': {'log_dir': '/home/levtel/pr...,"{'job': {'id': '???', 'num': '???', 'name': '$...",{'scaler_features': {'metadata': {'fit_on': 't...,NaN,NaN,debug,unibo+reconstruction+residual_autoencoder_2376...,"{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...",picid.optimizer.default.DefaultOptimizer,Adam,StepLR,picid.data.datasources.unibo.loader.UNIBO21Loader,/home/levtel/projects/TabPHM/datasets,UNIBO21,SOC,/home/levtel/projects/TabPHM/datasets/phmd_cache,reconstruction,[features],"[5, 5, 5, 5, 5]","[1, 1, 1, 1, 1]",picid.model.autoencoder.wrapper.AutoencoderWra...,"[1, 2, 4, 8, 1]","[16, 32, 64, 64, 64]","[{'in': 2, 'out': 16, 'kernel': 7, 'stride': 2...",picid.data.datasets.hydra_concat_dataset.Hydra...,picid.data.datasets.sliding_window_batch_datas...,picid.data.datamodules.base.BaseDataModule,picid.callbacks.timer_callback.PipelineTimer,lightning.pytorch.callbacks.RichModelSummary,min,val/loss,lightning.pytorch.callbacks.EarlyStopping,NaN,NaN,NaN,min,/home/levtel/projects/TabPHM/artifacts/debug/r...,val/loss,lightning.pytorch.callbacks.ModelCheckpoint,epoch{epoch:03d}-val_loss{val/loss:.2f},NaN,NaN,NaN,NaN,lightning.pytorch.callbacks.LearningRateMonitor,epoch,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/datasets,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM,/home/levtel/projects/TabPHM,/home/levtel/projects/TabPHM/datasets/cache/UN...,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts,/home/levt


--- ⚠️ Columns with NaN Values, 24.832214765100673% has NaN values ---


,num_nans,percentage_nans
ckpt_path,56,100.000000
profiler_cfg,56,100.000000
subset_range,56,100.000000
callbacks.early_stopping.stopping_threshold,56,100.000000
callbacks.model_checkpoint.train_time_interval,56,100.000000
callbacks.model_checkpoint.save_on_train_epoch_end,56,100.000000
callbacks.model_checkpoint.every_n_train_steps,56,100.000000
callbacks.early_stopping.check_on_train_epoch_end,56,100.000000
callbacks.early_stopping.divergence_threshold,56,100.000000
callbacks.model_checkpoint.every_n_epochs,56,100.000000


Dropping 13 columns with more than 90% NaNs


### The aim is to find the columns over which the sweep has been done hence are not needed (because they are not part of the sweeped parameters)

In [48]:
def filter_columns(df, filters=None, exclude=False):
    """
    Return columns that contain any of the filter strings, or those that do not contain any of them.
    If filters is None, return all columns.
    If filters is a string, treat as a single filter.
    If filters is a list/tuple, match any of the filters.
    If exclude is True, return columns that do NOT contain any of the filter strings.
    """
    def _matches(col, filters):
        return any(f in col for f in filters)

    # Handle dict input for filters/exclude
    if isinstance(filters, dict):
        filters_ = filters.get("filters", None)
        exclude = filters.get("exclude", False)
    else:
        filters_ = filters

    if filters_ is None:
        return df.columns.tolist()
    if isinstance(filters_, str):
        filters_ = [filters_]

    if exclude:
        return [col for col in df.columns if not _matches(col, filters_)]
    else:
        return [col for col in df.columns if _matches(col, filters_)]

# df["task_definition.model.data_requirements.input_tensors"] elemets are lists of strings
def get_unique_values(df, columns):
    """
    Extract unique values from one or multiple columns in a DataFrame.
    Handles elements that are lists, strings, or other types.
    Args:
        df: pandas DataFrame
        columns: str or list of str, column(s) to extract unique values from
    Returns:
        Set of unique values
    """
    if isinstance(columns, str):
        columns = [columns]
    unique_values = set()
    for col in columns:
        series = df[col].dropna()
        for item in series:
            if isinstance(item, list):
                unique_values.update(item)
            elif isinstance(item, str):
                unique_values.add(item)
            elif pd.api.types.is_scalar(item):
                unique_values.add(item)
            # Optionally handle other types if needed
    return list(unique_values)



In [49]:

cols_to_drop_1 = filter_columns(df, filters=["callbacks.model_checkpoint.", "paths.", "paths."], exclude=False)


additional_remove_col = [] + cols_to_drop_1
df = df.drop(additional_remove_col, axis=1)
droped_columns.extend(additional_remove_col)

### Some columns that are important might have nan 

In [ ]:

# Drop na values if there are
#df = df.dropna(subset=["model.backbone._target_"])
# Reset index
df = df.reset_index(drop=True)

# Drop rows that 'callbacks.early_stopping.monitor' isna
print(
    f"Number of rows with callbacks.early_stopping.monitor = nan is {sum(df['callbacks.early_stopping.monitor'].isna())}"
)

Number of rows with callbacks.early_stopping.monitor = nan is 0


In [ ]:
from collections import defaultdict

## <<< MODIFIED: Returns the full metric name (e.g., 'val/loss')
def get_optimized_metric(df_subset, model_name, dataset_name):
    """Finds and prints the single metric being optimized."""
    metric_series = get_unique_values(df=df_subset, columns="callbacks.early_stopping.monitor")
    if not metric_series:
        raise ValueError(f"No metric found for {model_name} on {dataset_name}. Check 'callbacks.early_stopping.monitor' column.")
    assert len(metric_series) == 1, "There should be only one metric to optimize"
    full_metric_name = metric_series[0]
    print(f"Optimized metric for {model_name} on {dataset_name} is {full_metric_name}")
    return full_metric_name

def get_varying_hyperparameters(df_subset, config_cols, special_cols_to_exclude):
    """Identifies which hyperparameters have more than one unique value."""
    unique_values = {}
    for col in config_cols:
        try:
            cleaned_series = df_subset[col].dropna()
            if cleaned_series.empty:
                continue
            if isinstance(cleaned_series.iloc[0], list):
                unique_values[col] = cleaned_series.apply(tuple).unique().tolist()
            else:
                unique_values[col] = get_unique_values(df_subset, col)
        except Exception as e:
            print(f"Attention: Could not process column '{col}': {e}")
            
    varying_params = {k: v for k, v in unique_values.items() if len(v) > 1}
    for col in special_cols_to_exclude:
        varying_params.pop(col, None)
    return varying_params

def aggregate_and_find_best(df_subset, aggregation_cols, all_performance_cols, optimization_col):
    """
    Performs groupby aggregation for all specified metrics and finds the best run based on the optimization metric.
    """
    subset_copy = df_subset.copy()
    
    for col in aggregation_cols:
        if col in subset_copy.columns and not subset_copy[col].dropna().empty and isinstance(subset_copy[col].dropna().iloc[0], list):
            subset_copy.loc[:, col] = subset_copy[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

    agg_dict = {col: ["mean", "std"] for col in all_performance_cols}
    aggregated_df = subset_copy.groupby(aggregation_cols).agg(agg_dict).reset_index()

    mode = subset_copy["callbacks.early_stopping.mode"].unique()[0]
    is_ascending = (mode == "min")

    sorted_results = aggregated_df.sort_values(by=(optimization_col, "mean"), ascending=is_ascending)
    
    best_result_row = sorted_results.head(1)
    
    return sorted_results, best_result_row

# ===================================================================
#  2. MAIN ANALYSIS FUNCTION
# ===================================================================

## <<< MODIFIED: Added 'metric_prefixes' parameter
def analyze_results(df, config_columns, droped_columns, reporting_metrics, metric_prefixes):
    """
    Analyzes experiment results from a DataFrame to find the best hyperparameters and report on specified metrics.
    """
    config_columns = [col for col in config_columns if col not in droped_columns]
    unique_models = get_unique_values(df, "model._target_")
    unique_datasets = get_unique_values(df, "datasource.data_name")
    special_columns = ["datasource.parameters.data_seed"]

    all_results = defaultdict(lambda: defaultdict(dict))

    for dataset in unique_datasets:
        for model in unique_models:
            subset = df[(df["datasource.data_name"] == dataset) & (df["model._target_"] == model)].copy()

            if subset.empty:
                print(f"---------\nNo results for {model} on {dataset}\n---------")
                continue
            
            subset.loc[:, 'Model'] = model
            
            # --- Step 1: Get key parameters and columns ---
            optimization_col = get_optimized_metric(subset, model, dataset)
            
            ## <<< MODIFIED: Create a list of all performance columns using the prefixes
            all_performance_cols = [
                f"{prefix}{metric}"
                for prefix in metric_prefixes
                for metric in reporting_metrics
                if f"{prefix}{metric}" in subset.columns
            ]
            
            varying_hyperparams = get_varying_hyperparameters(subset, config_columns, special_columns)
            aggregation_cols = ["Model"] + list(varying_hyperparams.keys())
            
            # --- Step 2: Print varying hyperparameters ---
            print(f"Hyperparameters that vary for {model} on {dataset}:")
            if not varying_hyperparams:
                print("None.")
            else:
                for col, unique in varying_hyperparams.items():
                    print(f"{col}: {unique}")
            print("---------")

            # --- Step 3: Aggregate results ---
            sorted_aggregated, best_result = aggregate_and_find_best(subset, aggregation_cols, all_performance_cols, optimization_col)
            
            # --- Step 4: Store all the collected results neatly ---
            ## <<< MODIFIED: Loop through prefixes and metrics to build the nested results dictionary
            mode = subset["callbacks.early_stopping.mode"].unique()[0]
            performance_results = defaultdict(dict)
            
            for metric in reporting_metrics:
                for prefix in metric_prefixes:
                    col_name = f"{prefix}{metric}"
                    prefix_key = prefix.strip('/') # 'val' or 'test'

                    if (col_name, "mean") in best_result.columns:
                        mean_val = best_result[(col_name, "mean")].values[0]
                        std_val = best_result[(col_name, "std")].values[0]

                        if col_name == optimization_col and mode == "max":
                            mean_val *= 100
                            std_val *= 100
                        
                        performance_results[metric][prefix_key] = {"mean": mean_val, "std": std_val}

            all_results[dataset][model]['best_performance'] = {
                "optimized_on": {"metric": optimization_col, "strategy": mode},
                "metrics": performance_results
            }
            all_results[dataset][model]['avg_epoch_time'] = {"mean": subset['AvgTime/train_epoch_mean'].mean(), "std": subset['AvgTime/train_epoch_mean'].std()}
            all_results[dataset][model]['avg_runtime'] = {"mean": subset["_runtime"].mean(), "std": subset["_runtime"].std()}
            all_results[dataset][model]['sorted_aggregated_results'] = sorted_aggregated
            all_results[dataset][model]['non_aggregated_df'] = subset.copy()
            
    return all_results

# ===================================================================
#  3. USAGE EXAMPLE
# ===================================================================

# ASSUMPTIONS: Your df, config_columns, droped_columns, and get_unique_values are defined

# 1. Define the prefixes for the metrics (e.g., for validation and test sets)
## <<< NEW: Define prefixes
metric_prefixes = ["val/", "test/"]

# 2. Define the base names of the metrics you want in the final report
reporting_metrics = ["loss","mse", "mae"] 

# 3. Run the analysis by calling the main function
all_results = analyze_results(
    df=df,
    config_columns=config_columns,
    droped_columns=droped_columns,
    reporting_metrics=reporting_metrics,
    metric_prefixes=metric_prefixes
)

print("\n✅ Analysis complete. Results are stored in the 'all_results' dictionary.")



Optimized metric for picid.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21 is val/loss
Hyperparameters that vary for picid.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21:
optimizer.scheduler.scheduler_params.step_size: [10, 20]
optimizer.parameters.lr: [0.01, 0.001, 0.0005]
model.kernels: [(5, 5, 5, 5, 5), (7, 7, 5, 5, 5)]
model.strides: [(1, 1, 1, 1, 1), (2, 2, 1, 1, 1)]
model.dilations: [(1, 2, 4, 8, 1), (1, 1, 1, 1, 1)]
datamodule.val_batch_size: [16, 1024]
datamodule.test_batch_size: [16, 1024]
datamodule.train_batch_size: [256, 128, 512, 1024, 16, 28]
---------
Optimized metric for picid.model.autoencoder.wrapper.AutoencoderWrapper on PRONOSTIA is val/loss
Hyperparameters that vary for picid.model.autoencoder.wrapper.AutoencoderWrapper on PRONOSTIA:
optimizer.scheduler.scheduler_params.step_size: [10, 20]
model.kernels: [(7, 7, 5, 5, 5), (5, 5, 5, 5, 5)]
model.strides: [(2, 2, 1, 1, 1), (1, 1, 1, 1, 1)]
model.dilations: [(1, 1, 1, 1, 1), (1, 2, 4, 8, 1)]
---------

In [88]:
#  pring for all all_results[dataset][model]['best_performance']
all_results['UNIBO21']['picid.model.autoencoder.wrapper.AutoencoderWrapper']['best_performance']


{'optimized_on': {'metric': 'val/loss', 'strategy': 'min'},
 'metrics': defaultdict(dict,
             {'loss': {'val': {'mean': np.float64(9.187756688788796e-08),
                'std': np.float64(nan)},
               'test': {'mean': np.float64(9.187756688788796e-08),
                'std': np.float64(nan)}},
              'mse': {'val': {'mean': np.float64(2.7563271487451857e-07),
                'std': np.float64(nan)},
               'test': {'mean': np.float64(2.7563271487451857e-07),
                'std': np.float64(nan)}},
              'mae': {'val': {'mean': np.float64(0.0001792862021829933),
                'std': np.float64(nan)},
               'test': {'mean': np.float64(0.0001792862021829933),
                'std': np.float64(nan)}}})}

In [92]:
def create_summary_table(all_results):
    """
    Processes the 'all_results' dictionary and creates a pivoted summary DataFrame.

    Args:
        all_results (dict): The nested dictionary generated by the analyze_results function.

    Returns:
        pandas.DataFrame: A pivoted summary table with models as rows and
                          dataset/metrics as columns.
    """
    records = []
    # Step 1: Flatten the nested dictionary into a list of records
    for dataset, models in all_results.items():
        for model, results in models.items():
            # Check if performance data exists for this entry
            if 'best_performance' in results and 'metrics' in results['best_performance']:
                metrics_data = results['best_performance']['metrics']
                for metric_name, prefixes in metrics_data.items():
                    for prefix, values in prefixes.items():
                        mean = values.get('mean', float('nan'))
                        std = values.get('std', float('nan'))

                        # Format the value as a "mean ± std" string, rounded to 2 decimal places
                        formatted_value = f"{mean:.8f} ± {std:.8f}"

                        records.append({
                            'Dataset': dataset,
                            'Model': model,
                            'Metric': f"{prefix}/{metric_name}",
                            'Value': formatted_value
                        })

    if not records:
        print("No performance data found to create a summary table.")
        return pd.DataFrame()

    # Step 2: Create a long-format DataFrame from the records
    long_df = pd.DataFrame(records)

    # Step 3: Pivot the DataFrame to create the wide-format summary table
    summary_table = long_df.pivot_table(
        index='Model',
        columns=['Dataset', 'Metric'],
        values='Value',
        aggfunc='first'  # Each combination is unique, so 'first' is appropriate
    )
    
    # Step 4: Clean up the table for better readability
    summary_table.sort_index(axis=1, inplace=True) # Sort columns alphabetically
    summary_table.fillna('-', inplace=True)      # Fill missing values

    return summary_table

In [93]:
# ===================================================================
#  4. CREATE AND DISPLAY THE FINAL SUMMARY TABLE
# ===================================================================

# Assuming 'all_results' has been generated by the analyze_results function

# 1. Create the pivoted summary table
summary_df = create_summary_table(all_results)

# 2. Display the final table
print("\n📊 Final Performance Summary Table:")
summary_df


📊 Final Performance Summary Table:


Dataset                                                    PRONOSTIA  \
Metric                                                     test/loss   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00014298 ± nan   

Dataset                                                               \
Metric                                                      test/mae   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00737906 ± nan   

Dataset                                                               \
Metric                                                      test/mse   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00028595 ± nan   

Dataset                                                               \
Metric                                                      val/loss   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00014298 ± nan   

Dataset                                                               \
Metric                                                       val/mae   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00737906 ± nan   

Dataset                                                               \
Metric                                                       val/mse   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00028595 ± nan   

Dataset                                                      UNIBO21  \
Metric                                                     test/loss   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00000009 ± nan   

Dataset                                                               \
Metric                                                      test/mae   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00017929 ± nan   

Dataset                                                               \
Metric                                                      test/mse   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00000028 ± nan   

Dataset                                                               \
Metric                                                      val/loss   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00000009 ± nan   

Dataset                                                               \
Metric                                                       val/mae   
Model                                                                  
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00017929 ± nan   

Dataset                                                               
Metric                                                       val/mse  
Model                                                                 
picid.model.autoencoder.wrapper.AutoencoderWrapper  0.00000028 ± nan

In [77]:
all_results['UNIBO21']['picid.model.autoencoder.wrapper.AutoencoderWrapper']['best_performance']

{'optimized_on': {'metric': 'val/loss', 'strategy': 'min'},
 'metrics': defaultdict(dict,
             {'loss': {'val': {'mean': np.float64(9.187756688788796e-08),
                'std': np.float64(nan)},
               'test': {'mean': np.float64(9.187756688788796e-08),
                'std': np.float64(nan)}},
              'mse': {'val': {'mean': np.float64(2.7563271487451857e-07),
                'std': np.float64(nan)},
               'test': {'mean': np.float64(2.7563271487451857e-07),
                'std': np.float64(nan)}},
              'mae': {'val': {'mean': np.float64(0.0001792862021829933),
                'std': np.float64(nan)},
               'test': {'mean': np.float64(0.0001792862021829933),
                'std': np.float64(nan)}}})}

In [79]:
all_results['UNIBO21']['picid.model.autoencoder.wrapper.AutoencoderWrapper']['sorted_aggregated_results']

Model  \
                                                        
1   picid.model.autoencoder.wrapper.AutoencoderWra...   
28  picid.model.autoencoder.wrapper.AutoencoderWra...   
8   picid.model.autoencoder.wrapper.AutoencoderWra...   
36  picid.model.autoencoder.wrapper.AutoencoderWra...   
0   picid.model.autoencoder.wrapper.AutoencoderWra...   
9   picid.model.autoencoder.wrapper.AutoencoderWra...   
2   picid.model.autoencoder.wrapper.AutoencoderWra...   
30  picid.model.autoencoder.wrapper.AutoencoderWra...   
10  picid.model.autoencoder.wrapper.AutoencoderWra...   
29  picid.model.autoencoder.wrapper.AutoencoderWra...   
37  picid.model.autoencoder.wrapper.AutoencoderWra...   
31  picid.model.autoencoder.wrapper.AutoencoderWra...   
3   picid.model.autoencoder.wrapper.AutoencoderWra...   
11  picid.model.autoencoder.wrapper.AutoencoderWra...   
38  picid.model.autoencoder.wrapper.AutoencoderWra...   
39  picid.model.autoencoder.wrapper.AutoencoderWra...   
41  picid.model.autoencoder.wrapper.AutoencoderWra...   
40  picid.model.autoencoder.wrapper.AutoencoderWra...   
6   picid.model.autoencoder.wrapper.AutoencoderWra...   
13  picid.model.autoencoder.wrapper.AutoencoderWra...   
14  picid.model.autoencoder.wrapper.AutoencoderWra...   
34  picid.model.autoencoder.wrapper.AutoencoderWra...   
12  picid.model.autoencoder.wrapper.AutoencoderWra...   
33  picid.model.autoencoder.wrapper.AutoencoderWra...   
5   picid.model.autoencoder.wrapper.AutoencoderWra...   
7   picid.model.autoencoder.wrapper.AutoencoderWra...   
15  picid.model.autoencoder.wrapper.AutoencoderWra...   
42  picid.model.autoencoder.wrapper.AutoencoderWra...   
35  picid.model.autoencoder.wrapper.AutoencoderWra...   
43  picid.model.autoencoder.wrapper.AutoencoderWra...   
32  picid.model.autoencoder.wrapper.AutoencoderWra...   
4   picid.model.autoencoder.wrapper.AutoencoderWra...   
26  picid.model.autoencoder.wrapper.AutoencoderWra...   
18  picid.model.autoencoder.wrapper.AutoencoderWra...   
21  picid.model.autoencoder.wrapper.AutoencoderWra...   
25  picid.model.autoencoder.wrapper.AutoencoderWra...   
27  picid.model.autoencoder.wrapper.AutoencoderWra...   
24  picid.model.autoencoder.wrapper.AutoencoderWra...   
19  picid.model.autoencoder.wrapper.AutoencoderWra...   
51  picid.model.autoencoder.wrapper.AutoencoderWra...   
50  picid.model.autoencoder.wrapper.AutoencoderWra...   
20  picid.model.autoencoder.wrapper.AutoencoderWra...   
48  picid.model.autoencoder.wrapper.AutoencoderWra...   
49  picid.model.autoencoder.wrapper.AutoencoderWra...   
47  picid.model.autoencoder.wrapper.AutoencoderWra...   
44  picid.model.autoencoder.wrapper.AutoencoderWra...   
46  picid.model.autoencoder.wrapper.AutoencoderWra...   
45  picid.model.autoencoder.wrapper.AutoencoderWra...   
16  picid.model.autoencoder.wrapper.AutoencoderWra...   
17  picid.model.autoencoder.wrapper.AutoencoderWra...   
22  picid.model.autoencoder.wrapper.AutoencoderWra...   
23  picid.model.autoencoder.wrapper.AutoencoderWra...   

   optimizer.scheduler.scheduler_params.step_size optimizer.parameters.lr  \
                                                                            
1                                              10                  0.0005   
28                                             20                  0.0005   
8                                              10                  0.0010   
36                                             20                  0.0010   
0                                              10                  0.0005   
9                                              10                  0.0010   
2                                              10                  0.0005   
30                                             20                  0.0005   
10                                             10                  0.0010   
29                                             20                  0.0005   
37                                           